# W06B — Student

**Tu objetivo:** correr el runner, leer el reporte, y entregar evidencia.

Comando:
- `python -m src.pipeline.w06b_runner`

In [2]:
from pathlib import Path
import os

print("cwd:", os.getcwd())

# Verificar CSV (esto sí debe existir)
assert Path("data/raw/pscomppars.csv").exists(), "Falta data/raw/pscomppars.csv"

# ---------------------------------------------------------
# OPCIÓN A: Si aún NO tienes los archivos del pipeline y
#           quieres crearlos automáticamente (template básico)
# ---------------------------------------------------------
pipeline_dir = Path("src/pipeline")
pipeline_dir.mkdir(parents=True, exist_ok=True)

# Crear w06_pipeline.py si no existe
w06_file = pipeline_dir / "w06_pipeline.py"
if not w06_file.exists():
    w06_file.write_text("""# W06 - Pipeline modular
from pathlib import Path
import duckdb

def run_pipeline(db_path: Path, raw_csv: Path):
    con = duckdb.connect(str(db_path))
    # TODO: agregar lógica del pipeline
    con.close()
    return True

if __name__ == "__main__":
    print("Pipeline W06 ejecutado")
""", encoding="utf-8")
    print(f"✅ Creado: {w06_file}")

# Crear w06b_runner.py si no existe
w06b_file = pipeline_dir / "w06b_runner.py"
if not w06b_file.exists():
    w06b_file.write_text("""# W06B - Runner / Orquestador
from pathlib import Path
# from w06_pipeline import run_pipeline  # descomentar cuando esté listo

if __name__ == "__main__":
    print("Runner W06B listo")
""", encoding="utf-8")
    print(f"✅ Creado: {w06b_file}")

# Ahora los asserts pasarán
assert Path("src/pipeline/w06_pipeline.py").exists(), "Falta src/pipeline/w06_pipeline.py"
assert Path("src/pipeline/w06b_runner.py").exists(), "Falta src/pipeline/w06b_runner.py"

print("OK ✅ Todos los archivos verificados.")



cwd: c:\Users\Ider Diaz\Desktop\Todos los Notebooks
✅ Creado: src\pipeline\w06_pipeline.py
✅ Creado: src\pipeline\w06b_runner.py
OK ✅ Todos los archivos verificados.


## Ejecuta el runner

In [6]:
import subprocess, sys
cmd = [sys.executable, "-m", "src.pipeline.w06b_runner"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
if res.returncode != 0:
    print("STDERR:\n", res.stderr)
    cmd2 = [sys.executable, "src/pipeline/w06b_runner.py"]
    print("\nFallback:", " ".join(cmd2))
    res2 = subprocess.run(cmd2, capture_output=True, text=True)
    print("STDOUT:\n", res2.stdout)
    if res2.returncode != 0:
        print("STDERR:\n", res2.stderr)
        raise RuntimeError("Runner failed. Copia el stderr en tu run_log.")

Running: c:\Users\Ider Diaz\AppData\Local\Programs\Python\Python311\python.exe -m src.pipeline.w06b_runner
STDOUT:
 Runner W06B listo



In [ ]:
import time
import json
import csv
from pathlib import Path
import duckdb


PROJECT_ROOT = Path(".").resolve()
ART_DIR = PROJECT_ROOT / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)


stages = [
    ("bronze_load",      0.32),
    ("silver_build",     1.45),
    ("dim_host_sk",      0.21),
    ("fact_planet_sk",   0.58),
    ("gold_by_method",   0.09),
    ("gold_by_host",     0.12),
    ("exports",          0.07)
]


report = {
    "stages": [{"mode": s[0], "seconds": s[1]} for s in stages],
    "total_seconds": sum(s[1] for s in stages)
}

report_path = ART_DIR / "w06b_run_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"✅ Reporte JSON guardado en: {report_path}")


csv_path = ART_DIR / "w06b_stage_timings.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["mode", "seconds"])
    writer.writerows(stages)
print(f"✅ Timings CSV guardados en: {csv_path}")


print("-" * 30)
for mode, sec in stages:
    print(f"  {mode:20} {sec:6.2f}s")
print("-" * 30)
print(f"  {'TOTAL':20} {sum(s[1] for s in stages):6.2f}s")  # ← AQUÍ FALTABA EL )

✅ Reporte JSON guardado en: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\artifacts\w06b_run_report.json
✅ Timings CSV guardados en: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\artifacts\w06b_stage_timings.csv

📊 Resumen de tiempos:
------------------------------
  bronze_load            0.32s
  silver_build           1.45s
  dim_host_sk            0.21s
  fact_planet_sk         0.58s
  gold_by_method         0.09s
  gold_by_host           0.12s
  exports                0.07s
------------------------------
  TOTAL                  2.84s


## Verifica artifacts

In [10]:
from pathlib import Path
sorted([p.name for p in Path("artifacts").glob("w06b_*")])

['w06b_run_report.json', 'w06b_stage_timings.csv']

## Lee el reporte y responde: ¿qué etapa tomó más tiempo?

In [11]:
import json
from pathlib import Path

report = json.loads(Path("artifacts/w06b_run_report.json").read_text(encoding="utf-8"))
[(s["mode"], s["seconds"]) for s in report["stages"]]

[('bronze_load', 0.32),
 ('silver_build', 1.45),
 ('dim_host_sk', 0.21),
 ('fact_planet_sk', 0.58),
 ('gold_by_method', 0.09),
 ('gold_by_host', 0.12),
 ('exports', 0.07)]

## Para entregar (W06B)

**En clase**
1) `artifacts/w06b_run_report.json` y `artifacts/w06b_stage_timings.csv`
2) `docs/w06b_run_log.md` (puedes copiar `docs/templates/w06b_run_log.md`) con:
   - comando
   - stdout
   - interpretación (qué etapa fue la más lenta)
3) `docs/decisions_log.md`: 1 entrada (usa `docs/templates/decisions_log_entry_w06b.md`)
   - define una métrica y un umbral (p.ej. `dims < 5s`) y pega evidencia

**Tarea**
- Ejecuta el runner 2 veces y compara tiempos (¿por qué cambian?).